# Homework Problem 1

**Date/Time:** 28 Feb 2026 18:22:45 UTC

**Compute:**
- 3rd-body acceleration for sun
- 3rd-body acceleration for moon
- Drag acceleration with F10 = 100  
  *(Remember: model uses F10scaled = F10/100)*
- Solar radiation pressure
- *(For extra fun: Gravitational acceleration for n=m=4)*

**Compute total acceleration on SV**

---

## Homework Inputs

**Vector (Earth-Fixed, Inertial)**

| Parameter | Value          | Units    | Parameter | Value      | Units |
|-----------|----------------|----------|-----------|------------|-------|
| X         | 5907119        | m        | LATC      | 25.82091   | deg   |
| Y         | -1465029       | m        | LATD      | 25.87529   | deg   |
| Z         | 2944866        | m        | LON       | 346.0693   | deg   |
| XD        | 935.1631       | m/sec    | ALTD      | 387.0463   | km    |
| YD        | 7414.308       | m/sec    |           |            |       |
| ZD        | 1901.963       | m/sec    |           |            |       |

- **LATC** = Geocentric latitude  
- **LATD** = Geodetic latitude  
- **LON** = Longitude  
- **ALTD** = Geodetic Altitude  

**Notes:**  
- Vector is inertial, but corresponds to ECEF.  
- No need to do RNP transformation for this case.  
- Pretend that sun/moon vectors are in this frame.

---

## More Inputs (Constants)

- Drag coefficient: \( C_d = 2.0 \)
- Drag Area = \( 10 \, m^2 \)
- Radiation coefficient: \( C_R = 1.41 \)
- Solar Pressure Area = \( 20 \, m^2 \)
- Vehicle mass = \( 1000 \, kg \)
- \( \omega_\oplus = 72.921151467 \times 10^{-6} \) rad/sec
- 1 nautical mile = 1.852 km (exactly)
- \( \mu_{moon} = \mu_\oplus / 81.3005764441083 \)
- \( \mu_{sun} = \mu_\oplus \cdot 332946.09358859973 \)

<img src="A_3rd_body.png" alt="A_3rd_body" width=600/>

In [6]:
from math import *
from standards import *

#### given info
pos_ = Vector3(5907119, -1465029, 2944866)
vel_ = Vector3(935.1631, 7414.308, 1901.963)
LATC = 25.82091
LATD = 25.87529
LON = 346.0693
ALTD = 387.0463

UTC_year = 2026
UTC_month = 2
UTC_day = 28
UTC_hour = 18
UTC_minute = 22
UTC_second = 45

mu_earth = 3.986004418e14


#### Compute JD_TT
J_date_midnight = (
    floor((1461*(UTC_year+4800+(UTC_month-14)/12))/4)
    +floor((367*(UTC_month-2-12*((UTC_month-14)/12)))/12)
    -floor((3*((UTC_year+4900+(UTC_month-14)/12)/100))/4)
    +UTC_day-32075
)
d = UTC_hour/24+UTC_minute/1440+UTC_second/86400-0.5
J_date = J_date_midnight + d
MJD = J_date - 2400000.5
T = 2000 + (MJD - 51544.03)/365.242199
UT2_UT1 = 0.022*sin(2*pi*T) - 0.012*cos(2*pi*T) - 0.006*sin(4*pi*T) + 0.007*cos(4*pi*T)
TAI_UTC = 37
TT_UTC = TAI_UTC + 32.184
UT1_UTC = 0.0640+0.00003*(MJD-61091) - (UT2_UT1)
Sec_From_J2000 = round((J_date - 2451545.0)*86400)
TAI_Sec = Sec_From_J2000 + TAI_UTC
JD_TT = J_date + TT_UTC/86400


#### compute sun vector 
n = JD_TT-2451545
L = (280.460 + 0.9856474*n)%360
g = (357.528 + 0.9856003*n)%360
Ecliptic_lon = L + 1.915*sin(radians(g))+0.020*sin(2*radians(g))
Ecliptic_lat = 0
Obliqity_of_eliptic = 23.439 - 0.0000004*n
R = 1.00014-0.01671*cos(radians(g))-0.00014*cos(2*radians(g))
x = R*cos(radians(Ecliptic_lon))
y = R*cos(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
z = R*sin(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
m_in_au = 149597870700
sun_coordinates = Vector3(x*m_in_au, y*m_in_au, z*m_in_au)
sun_ = sun_coordinates.get_np_vector()


#### compute acceleration due to sun
r_rel_ = sun_ - pos_.get_np_vector()
u_sun = mu_earth * 332946.09358859973
a_sun = u_sun * (r_rel_/(np.linalg.norm(r_rel_)**3) - sun_/(np.linalg.norm(sun_)**3))
print("Sun Point Mass Gravity:")
print(a_sun)


Sun Point Mass Gravity:
[[ 4.05357611e-07]
 [-1.54019658e-07]
 [-2.12742334e-07]]
